# 2025-03-16: Revisiting the Transitive Matching Strategy

## Prelude

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
from os import environ
import apfel as apfel
import statsmodels.api as sm

In [ ]:
dataset = pl.read_parquet(f"{environ['DATA_PATH_BASE']}/v2/babylm/0310/parsed-roberta.pos-lemma.parquet")

In [ ]:
dataset

## Extract Transitive Verbs (New)

Sentences with "'" (apostrophe) are not tokenized correctly, before I fix that, I filter them out first.

In [ ]:
dataset.select(
    pl.col("text").str.contains("'").value_counts().struct.rename_fields(["contains_apostrophe", "count"]).struct.unnest(),
)

In [ ]:
dataset = dataset.filter(
    pl.col("parse").is_not_null()
    & ~pl.col("text").str.contains("'")
)

New transitive verb extraction selection rule:

1. Marked as `VERB` by spaCy POS tagger
2. Has a direct object `dobj`

**Why no `nsubjpass`?**

Explicitly I didn't include the passive subjects, because passive mood is also a competing construction.
Consider the word *grow*.

- "Milk grows bones"
- "Bones are grown by milk"
- "Milk makes bones grow"

Alternatively,
- "Farmers grow crops"
- "Crops are grown by farmers"
- "Farmers make crops grow"

In [ ]:
from verb_overgen.prep.lib.pattern_matching import match_transitive

dataset_trans_verb = dataset.filter(
    pl.col("parse").is_not_null(),
    ~pl.col("text").str.contains("'"),
).select(
    pl.col("index"),
    trans_verb_index := match_transitive(pl.col("parse"), pl.col("pos")).alias("trans_verb_index"),
    trans_verb := pl.col("lemma").list.gather(trans_verb_index - 1).alias("trans_verb"),
).filter(
    pl.col("trans_verb_index").list.len() > 0
)

In [ ]:
df_all_trans_verb = dataset_trans_verb.select(
    pl.col("trans_verb").explode().value_counts().struct.unnest()
).sort(pl.col("count"), pl.col("trans_verb"), descending=[True, False])

df_all_trans_verb

In [ ]:
df_all_verb = dataset.filter(
    pl.col("parse").is_not_null(),
    ~pl.col("text").str.contains("'"),
).select(
    imperative(
        verb_index := pl.col("pos").list.eval(pl.element().eq("VERB").arg_true()),
        pl.col("lemma").list.gather(verb_index).explode()
    ).alias("verb")
).filter(
    pl.col("verb").is_not_null()
).select(
    pl.col("verb").value_counts().struct.unnest()
).sort(
    pl.col("count"), pl.col("verb"), descending=[True, False]
)

In [ ]:
df_all_verb.filter(
    ~pl.col("verb").is_in(df_all_trans_verb.get_column("trans_verb"))
)#.write_csv(
#    f"/Users/tomyxw/Documents/Projects/work/verb-overgen/repo/src/verb_overgen/prep/asset/candidate_selection/verb_not_transitive.csv",
#)

## Case Study

### *laugh*

Some common verbs do not have direct objects at all:

| verb | occurence |
| :--- | :--- |
| stay | 281 |
| laugh | 50 |

In [ ]:
dataset_with_verb = dataset.with_columns(
    imperative(
        verb_index := pl.col("pos").list.eval(pl.element().eq("VERB").arg_true()),
        pl.col("lemma").list.gather(verb_index)
    ).alias("verb")
)

In [ ]:
from verb_overgen.prep.lib.graph import draw_parse_graph, create_parse_graph, add_attr_to_graph

In [ ]:
import re
import rich
from rich.table import Table

def display_word(word):
    table = Table(title="Examples of the verb [i]{word}[/i]".format(word=word))
    table.add_column("index", justify="right", style="cyan")
    table.add_column("text", justify="left", overflow="fold")

    df = dataset_with_verb.filter(
            pl.col("verb").list.contains(word)
        )


    for i, row in enumerate(df.iter_rows(named=True)):
        text = row["text"]
        text = re.sub(rf"({word}\w*)", r"[u b]\g<0>[/]", text)

        table.add_row(
            str(row["index"]),
            text,
        )

    display(table)

In [ ]:
display_word("laugh")

In [ ]:
from collections.abc import Callable

In [ ]:
def for_each(src: pl.Expr, target: pl.Expr, op: Callable[[pl.Expr, pl.Expr], pl.Expr]) -> pl.Expr:
    return pl.concat_list(pl.struct(src, target)).list.eval(
        op(pl.element().struct[0].explode(), pl.element().struct[1])
    )

In [ ]:
def get_dataset_with_sample_verb(
    sample_verb: str,
) -> pl.DataFrame:
    dataset_with_sample_verb = dataset_with_verb.filter(pl.col("verb").list.contains(sample_verb))

    dataset_with_sample_verb_rel = dataset_with_sample_verb.with_columns(
        sample_verb_idx := imperative(
            is_sample := pl.col("lemma").list.eval(pl.element().eq(sample_verb).arg_true()),
            is_verb := pl.col("pos").list.eval(pl.element().eq("VERB").arg_true()),
            is_sample.list.set_intersection(is_verb).alias("sample_verb_idx"), # type: ignore
        )
    ).explode(
        "sample_verb_idx"
    ).with_columns(
        sample_verb_in_edge := for_each(
            pl.col("parse").struct.field("arc"),
            pl.col("sample_verb_idx"),
            lambda src, target: (src == target + 1).arg_true()
        ).alias("sample_verb_in_edge"),
        pl.col("parse").struct.field("rel").list.gather(sample_verb_in_edge).alias("sample_verb_child_rel"),
        pl.col("parse").struct.field("rel").list.gather(
            pl.concat_list(pl.col("sample_verb_idx"))
        ).list.explode().alias("sample_verb_parent_rel"),
        sample_verb_parent_idx := pl.col("parse").struct.field("arc").list.gather(pl.concat_list(pl.col("sample_verb_idx"))).list.first().alias("sample_verb_parent_idx"),
        sample_verb_parent := pl.when(sample_verb_parent_idx == 0)
            .then(None)
            .otherwise(pl.col("lemma").list.gather(pl.concat_list(sample_verb_parent_idx - 1)).list.first())
            .alias("sample_verb_parent"),
    )

    return dataset_with_sample_verb_rel



In [ ]:
df = get_dataset_with_sample_verb("laugh")

with pl.Config(set_tbl_rows=-1, set_fmt_table_cell_list_len=-1, set_tbl_width_chars=-1, set_fmt_str_lengths=64):
    display(df.select("index", "text", "sample_verb_child_rel", "sample_verb_parent_rel"))

### *grow*

In [ ]:
display_word("grow")

In [ ]:
df = get_dataset_with_sample_verb("grow")

In [ ]:
def select_core_cols(df):
    return df.select("index", "text", "sample_verb_child_rel", "sample_verb_parent_rel")

In [ ]:
df_idx = set()

Passive usages

In [ ]:
with pl.Config(set_tbl_rows=-1, set_fmt_table_cell_list_len=-1, set_tbl_width_chars=-1, set_fmt_str_lengths=64):
    df_group = df.filter(
        passive := pl.col("sample_verb_child_rel").list.contains("nsubjpass")
    ).pipe(function=select_core_cols)

    display(df_group)

df_idx.update(df_group.get_column("index").to_list())


    # display(df.filter(pl.col("sample_verb_child_rel").list.contains("nsubj")).pipe(select_core_cols))

Transitive usages

In [ ]:
with pl.Config(set_tbl_rows=-1, set_fmt_table_cell_list_len=-1, set_tbl_width_chars=-1, set_fmt_str_lengths=64):
    df_group = df.filter(
        transitive := pl.col("sample_verb_child_rel").list.contains("dobj")
    ).pipe(select_core_cols)

    display(df_group)

df_idx.update(df_group.get_column("index").to_list())

Causative usages

In [ ]:
with pl.Config(set_tbl_rows=-1, set_fmt_table_cell_list_len=-1, set_tbl_width_chars=-1, set_fmt_str_lengths=64):
    df_group = (df.filter(
        causative := (pl.col("sample_verb_parent_rel").is_in(["ccomp", "xcomp"])
        & pl.col("sample_verb_parent").is_in(["make", "have", "let", "get"])
        & pl.col("sample_verb_child_rel").list.contains("nsubj")
        )
    ).pipe(function=select_core_cols))

    display(df_group)

df_idx.update(df_group.get_column("index").to_list())

Residual

Interesting case: are these two the same *grow*?

- "Bananas grow on trees"
- "Bananas are grown on trees"

In [ ]:
with pl.Config(set_tbl_rows=-1, set_fmt_table_cell_list_len=-1, set_tbl_width_chars=-1, set_fmt_str_lengths=64):
    df_group = (df.filter(
        ~pl.col("index").is_in(df_idx)
    ).pipe(function=select_core_cols))

    display(df_group)

## Bidgood Verb List

In [ ]:
def get_verb_construction_count(verb: str):
    return get_dataset_with_sample_verb(verb).select(
        pl.lit(verb).alias("verb"),
        passive.arg_true().count().alias("passive"),
        transitive.arg_true().count().alias("transitive"),
        causative.arg_true().count().alias("causative"),
        (~(passive | transitive | causative)).arg_true().count().alias("other"),
        pl.len().alias("total"),
    )

In [ ]:
get_verb_construction_count("break")

In [ ]:
dataset_verb_construction_count = pl.concat([get_verb_construction_count(verb) for verb in df_all_verb.filter(pl.col("count") >= 50).get_column("verb")])
from verb_overgen.prep.lib.lexical import BIDGOOD_VERBS

df_bidgood_v = pl.DataFrame([(category, word) for category, s in BIDGOOD_VERBS.items() for word in s], orient="row", schema=pl.Schema({
    "bidgood": pl.Enum([*BIDGOOD_VERBS.keys(), "non_bidgood"]),
    "verb": pl.String
}))
dataset_verb_construction_count = dataset_verb_construction_count.join(
    df_bidgood_v,
    left_on="verb",
    right_on="verb",
    how="left"
).with_columns(
    pl.when(pl.col("bidgood").is_null()).then(pl.lit("non_bidgood")).otherwise(pl.col("bidgood")).alias("bidgood")
)

In [ ]:
df_v = dataset_verb_construction_count.select(
    "verb",
    "bidgood",
    pl.col("transitive").add(pl.col("passive")).alias(name="transitive"),
    pl.col("causative").add(pl.col("other")).alias("intransitive"),
).filter(pl.col("bidgood") != "non_bidgood")

fig, ax = plt.subplots(figsize=(10, 10))

sns.scatterplot(
    data=df_v,
    ax=ax,
    x="transitive", y="intransitive",
    hue="bidgood",
    marker="X",
)

for (verb, _, count_trans, count_intrans, *_) in df_v.iter_rows():
    ax.text(
        count_trans, count_intrans,
        f"{verb}",
        fontsize=8,
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_aspect("equal")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Category")

plt.savefig("trans-intrans-bidgood-new.png")

![](trans-intrans-bidgood-new.png)

**Next step**

- Find more representative verbs
- Check its competing structures left in the "other" column (which currently goes to intransitive)
- Refine the rules

In [ ]:
get_dataset_with_sample_verb("take").filter(
        ~(passive | transitive | causative)
).sample(10).select("index", "text")

In [ ]:
with pl.Config(set_fmt_str_lengths=128):
    display(get_dataset_with_sample_verb("feed").filter(
        ~(passive | transitive | causative)
).select("index", "text"))

## Dependency Parsing Demo Ground

In [ ]:
idx = 207907

In [ ]:
item = dataset.filter(pl.col(name="index") == 141487).row(0, named=True)
item["text"]

In [ ]:
item = dataset.filter(pl.col(name="index") == idx).row(0, named=True)

graph = create_parse_graph(item["parse"])
add_attr_to_graph(graph, lemma=item["lemma"], pos=item["pos"])
fig = draw_parse_graph(graph)